# Checkerboard susceptibility figures (self-contained)

Run cells top to bottom. Each figure runs the QMC in parallel (`multiprocessing`) and plots inline. No CSV files. Edit params/fonts in the SETUP cell.

**Requires** the platform at `~/qmc-platform-master/pyqmc` (checkerboard.py + cpqmc.py).

In [ ]:
# --- SETUP (edit style + params here) ---
import os, sys
os.environ["OMP_NUM_THREADS"]="1"; os.environ["MKL_NUM_THREADS"]="1"
sys.path.insert(0, os.path.expanduser("~/qmc-platform-master/pyqmc"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from multiprocessing import Pool
from cpqmc import CPMC
import checkerboard as cb
%matplotlib inline

NPROC = min(os.cpu_count(), 32)     # cores to use (lower if sharing the node)
U, L, T0, T1 = 4.0, 6, -1.0, 0.3    # U, lattice L (6x6), t0=-1, t1=-t'(=+0.3)
NW, NEQ, NBLK, BP, DT = 160, 60, 40, 16, 0.05

# ---- FONT / STYLE (edit here) ----
FS_LABEL, FS_TICK, FS_TITLE, FS_LEG, MS = 18, 13, 16, 15, 10

def run_pair(args):
    nup, delta, seed = args
    K = cb.checkerboard_hopping(L, L, T0, T1, -delta)
    Fd = cb.nn_bond_factors(L, L)[1]; Fdxy = cb.diag_bond_factors(L, L)
    q = CPMC(L, L, nup, nup, U=U, dt=DT, nwalkers=NW, seed=seed, K=K, K_dn=None)
    r = cb.run_bp_chid_cb(q, {"d": Fd, "dxy": Fdxy}, nequil=NEQ, nblocks=NBLK, bp=BP)
    return (nup, delta, seed, r["chi_d_vertex"], r["chi_dxy_vertex"])

def run_chizz(args):
    nup, delta = args
    K = cb.checkerboard_hopping(L, L, T0, T1, -delta)
    q = CPMC(L, L, nup, nup, U=U, dt=DT, nwalkers=NW, seed=1, K=K, K_dn=None)
    r = q.run_bp_chi_spin(nequil=NEQ, nblocks=NBLK, bp=BP)
    return (nup, delta, np.array(r["chi_q"]), r["chi_pipi"])

print("ready. NPROC =", NPROC)

In [ ]:
# FIG 9 : pairing chi vs delta at n=0.78 (nup=14), 6 seeds  (~5 min)
jobs = [(14, d, s) for d in [0.0,0.1,0.2,0.3,0.4] for s in range(1,7)]
with Pool(NPROC) as p: res = p.map(run_pair, jobs)
df = pd.DataFrame(res, columns=["nup","delta","seed","chi_d","chi_dxy"])
g9 = df.groupby("delta").agg(d=("chi_d","mean"),de=("chi_d","sem"),
                             x=("chi_dxy","mean"),xe=("chi_dxy","sem")).reset_index()
fig,ax=plt.subplots(figsize=(7,5.5))
ax.errorbar(g9.delta,g9.d,g9.de,marker="o",ms=MS,lw=2,capsize=4,color="#1f77b4",label=r"$d_{x^2-y^2}$")
ax.errorbar(g9.delta,g9.x,g9.xe,marker="^",ms=MS,lw=2,capsize=4,color="#2ca02c",label=r"$d_{xy}$")
ax.axhline(0,color="gray",ls="--",lw=1)
ax.set_xlabel(r"Anisotropy $\delta$",fontsize=FS_LABEL)
ax.set_ylabel(r"$\chi^{\rm vertex}_{\zeta}$ (pairing)",fontsize=FS_LABEL-2)
ax.set_title(r"$6\times6$, $n=0.78$, $U=4$",fontsize=FS_TITLE)
ax.tick_params(labelsize=FS_TICK); ax.legend(fontsize=FS_LEG,frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# FIG 10 : pairing chi vs filling n at delta=0.4, 3 seeds  (~5 min)
jobs = [(nu,0.4,s) for nu in [6,8,10,12,14,16,18] for s in [1,2,3]]
with Pool(NPROC) as p: res = p.map(run_pair, jobs)
df = pd.DataFrame(res, columns=["nup","delta","seed","chi_d","chi_dxy"]); df["n"]=2*df.nup/(L*L)
g10 = df.groupby("n").agg(d=("chi_d","mean"),de=("chi_d","sem"),
                          x=("chi_dxy","mean"),xe=("chi_dxy","sem")).reset_index()
fig,ax=plt.subplots(figsize=(7,5.5))
ax.errorbar(g10.n,g10.d,g10.de,marker="o",ms=MS,lw=2,capsize=4,color="#1f77b4",label=r"$d_{x^2-y^2}$")
ax.errorbar(g10.n,g10.x,g10.xe,marker="^",ms=MS,lw=2,capsize=4,color="#2ca02c",label=r"$d_{xy}$")
ax.axhline(0,color="gray",ls="--",lw=1)
ax.set_xlabel(r"Filling $n$",fontsize=FS_LABEL)
ax.set_ylabel(r"$\chi^{\rm vertex}_{\zeta}$ (pairing)",fontsize=FS_LABEL-2)
ax.set_title(r"$6\times6$, $\delta=0.4$, $U=4$",fontsize=FS_TITLE)
ax.tick_params(labelsize=FS_TICK); ax.legend(fontsize=FS_LEG,frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# FIG 11 : magnetic chi_zz(q) BZ maps  (~2 min)
jobs = [(11,0.0),(11,0.4),(18,0.4)]
with Pool(len(jobs)) as p: res = p.map(run_chizz, jobs)
pi=np.pi
fig,axes=plt.subplots(1,3,figsize=(15,4.6))
for ax,(nup,delta,chi,cp) in zip(axes,res):
    m=np.fft.fftshift(chi)
    im=ax.imshow(m.T,origin="lower",cmap="magma",extent=[-pi,pi,-pi,pi],aspect="equal")
    fig.colorbar(im,ax=ax,shrink=0.8)
    ax.set_xticks([-pi,0,pi]); ax.set_yticks([-pi,0,pi])
    ax.set_xticklabels([r"$-\pi$","0",r"$\pi$"]); ax.set_yticklabels([r"$-\pi$","0",r"$\pi$"])
    ax.set_xlabel(r"$q_x$",fontsize=15)
    ax.set_title(rf"$\chi_{{zz}}(q)$: $n={2*nup/(L*L):.2f},\ \delta={delta}$",fontsize=14)
axes[0].set_ylabel(r"$q_y$",fontsize=15)
plt.tight_layout(); plt.show()

In [ ]:
# FIG 12 : staggered magnetic chi_zz(pi,pi) vs filling n at delta=0.4  (~5 min)
jobs = [(nu,0.4) for nu in [6,8,10,12,14,16,18]]
with Pool(NPROC) as p: res = p.map(run_chizz, jobs)
czn = sorted((2*nu/(L*L), cp) for (nu,delta,chi,cp) in res)
ns = [a for a,b in czn]; cz = [b for a,b in czn]
fig,ax=plt.subplots(figsize=(7,5.5))
ax.plot(ns,cz,marker="s",ms=MS,lw=2,color="#8c2d8c")
ax.set_xlabel(r"Filling $n$",fontsize=FS_LABEL)
ax.set_ylabel(r"$\chi_{zz}(\pi,\pi)$ (staggered magnetic)",fontsize=FS_LABEL-2)
ax.set_title(r"$6\times6$, $\delta=0.4$, $U=4$",fontsize=FS_TITLE)
ax.tick_params(labelsize=FS_TICK); plt.tight_layout(); plt.show()

In [ ]:
# FIG 13 : magnetism-vs-pairing overlay  (reuses g10 from Fig 10, ns/cz from Fig 12)
fig,ax=plt.subplots(figsize=(7.5,5.5))
l1,=ax.plot(ns,cz,marker="s",ms=9,lw=2,color="#8c2d8c",label=r"$\chi_{zz}(\pi,\pi)$ (magnetic)")
ax.set_xlabel(r"Filling $n$",fontsize=FS_LABEL)
ax.set_ylabel(r"$\chi_{zz}(\pi,\pi)$",fontsize=FS_LABEL-2,color="#8c2d8c")
ax.tick_params(axis="y",labelcolor="#8c2d8c",labelsize=FS_TICK); ax.tick_params(axis="x",labelsize=FS_TICK)
ax2=ax.twinx()
l2=ax2.errorbar(g10.n,g10.d,g10.de,marker="o",ms=9,lw=2,capsize=4,color="#1f77b4",label=r"$\chi_{d_{x^2-y^2}}$ (pairing)")
ax2.axhline(0,color="gray",ls="--",lw=0.8)
ax2.set_ylabel(r"$\chi^{\rm vertex}_{d_{x^2-y^2}}$",fontsize=FS_LABEL-2,color="#1f77b4")
ax2.tick_params(axis="y",labelcolor="#1f77b4",labelsize=FS_TICK)
ax.set_title(r"magnetism vs pairing, $\delta=0.4$",fontsize=FS_TITLE)
ax.legend([l1,l2],[l1.get_label(),l2.get_label()],fontsize=FS_LEG,frameon=False,loc="upper center")
plt.tight_layout(); plt.show()